In [2]:
# Following is originally copied from PyTorch RNN-T ASR Example:
# https://github.com/pytorch/audio/tree/820b383b3b21fc06e91631a5b1e6ea1557836216/examples/asr/librispeech_emformer_rnnt
import sys
sys.path.append('../')
import pathlib
from argparse import ArgumentParser
import yaml

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.plugins import DDPPlugin




In [3]:
ckpt = '../ckpt/giga_word_attn_train_sd0/latest.pth'

In [5]:

expr_dir = pathlib.Path("../exp")
config = yaml.load(open('../config/giga/giga_word_las_train.yaml', 'r'), Loader=yaml.FullLoader)
checkpoint_dir = expr_dir / "checkpoints"
checkpoint = ModelCheckpoint(
    checkpoint_dir,
    monitor="WER/val_att",
    mode="min",
    save_top_k=1,
    save_weights_only=True,
    verbose=True,
)
train_checkpoint = ModelCheckpoint(
    checkpoint_dir,
    monitor="Losses/train_loss",
    mode="min",
    save_top_k=1,
    save_weights_only=True,
    verbose=True,
)
callbacks = [
    checkpoint,
    train_checkpoint,
]
trainer = Trainer(
    default_root_dir=expr_dir,
    max_epochs=config['hparas']['epochs'],
    log_every_n_steps = 10,
    num_nodes=1,
    gpus=2,
    accelerator="gpu",
    limit_train_batches=0.1, 
    strategy=DDPPlugin(find_unused_parameters=False),
    val_check_interval=config['hparas']['valid_step'],
    gradient_clip_val=10.0,
    profiler="simple",
    callbacks=callbacks,
)
if config['model_name'] == 'RNNT':
    from src.giga_rnnt_lightning import RNNTModule
    model = RNNTModule(config)
elif config['model_name'] == 'LAS':
    from src.giga_las_lightning import LASModule
    model = LASModule(config)      



GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs


TypeError: __init__() missing 1 required positional argument: 'config'

In [ ]:
trainer.fit(model)
